In [ ]:
# 1. Spark Application Setup

from pyspark.sql import SparkSession
from pyspark.sql.functions import *
from pyspark.sql.types import *

# Initialize Spark Session

spark = (
    SparkSession.builder
    .appName("ECommerce Analytics Pipeline")
    .master("local[*]")
    .getOrCreate()
)

In [ ]:
# TASK 2: Data Ingestion & Validation

# Defining schema explicitly 
transaction_schema = StructType([
    StructField("transaction_id", StringType(), False),
    StructField("user_id", StringType(), True),
    StructField("product_id", StringType(), True),
    StructField("category", StringType(), True),
    StructField("price", DoubleType(), True),
    StructField("quantity", IntegerType(), True),
    StructField("transaction_timestamp", TimestampType(), True),
    StructField("platform", StringType(), True),
    StructField("country", StringType(), True)
])


# Reading the CSV File
df_raw = spark.read \
    .option("header", True) \
    .option("mode", "PERMISSIVE") \
    .option("timestampFormat", "yyyy-MM-dd HH:mm:ss") \
    .schema(transaction_schema) \
    .csv("transactions.csv")

# Identifying Invalid / Malformed Records
invalid_df = df_raw.filter(
    col("transaction_id").isNull() |
    col("user_id").isNull() |
    col("product_id").isNull() |
    col("category").isNull() |
    col("price").isNull() |
    col("quantity").isNull() |
    col("transaction_timestamp").isNull() |
    col("platform").isNull() |
    col("country").isNull()
)

# Extracting Valid Records
valid_df = df_raw.filter(
    col("transaction_id").isNotNull() &
    col("user_id").isNotNull() &
    col("product_id").isNotNull() &
    col("category").isNotNull() &
    col("price").isNotNull() &
    col("quantity").isNotNull() &
    col("transaction_timestamp").isNotNull() &
    col("platform").isNotNull() &
    col("country").isNotNull()
)

# Data Quality Reporting
total_count = df_raw.count()
valid_count = valid_df.count()
invalid_count = invalid_df.count()

# Counting total, valid, and invalid records for audit purposes
print("Total Records:", total_count)
print("Valid Records:", valid_count)
print("Invalid / Malformed Records:", invalid_count)

Total Records: 214
Valid Records: 197
Invalid / Malformed Records: 17


In [ ]:
# TASK 3: Data Cleaning & Standardization

# Drop missing rows
required_cols = ["transaction_id", "user_id", "product_id", "category", "price", "quantity", "transaction_timestamp"]
clean_df = valid_df.dropna(subset=required_cols)


# Remove Duplicate Records
clean_df = clean_df.dropDuplicates()


# Column Formatting and Type Casting
clean_df = clean_df.withColumn("category", trim(lower(col("category"))))
clean_df = clean_df.withColumn("price", col("price").cast("double")) \
                   .withColumn("quantity", col("quantity").cast("int")) \
                   .withColumn("transaction_timestamp",
                               to_timestamp(col("transaction_timestamp"), "yyyy-MM-dd HH:mm:ss"))

In [ ]:
# TASK 4: Partitioning & Performance Optimization

clean_df.rdd.getNumPartitions()
df_repart = clean_df.repartition(4)
clean_df.cache()
final_df = clean_df.coalesce(1)

In [ ]:
# TASK 5: Business Analytics & Aggregations

# 1. Total Revenue per Country
total_revenue=clean_df.groupby("country").agg({"price":"sum"})

# 2. Top 5 Performing Products
top_perfomer=clean_df.groupby("product_id").agg({"price":"sum"}).orderBy(desc("sum(price)")).show(5)

# 3. Average Order Value per Platform
avg_ord_pltform=clean_df.groupby("platform").agg({"price":"avg"}).orderBy(desc("avg(price)")).show()


# 4. Daily Transaction Trends
daily_trend_df = clean_df.withColumn("transaction_date", to_date(clean_df.transaction_timestamp))


daily_trend_df = daily_trend_df.groupBy("transaction_date") \
    .agg(count("transaction_id").alias("daily_transaction_count")) \
    .orderBy("transaction_date").show()

+----------+----------+
|product_id|sum(price)|
+----------+----------+
|   PROD076|   3999.98|
|   PROD046|   3599.98|
|   PROD092|   3399.98|
|   PROD032|   3199.98|
|   PROD086|   3199.98|
+----------+----------+
only showing top 5 rows
+--------+-----------------+
|platform|       avg(price)|
+--------+-----------------+
|     Web|602.3525510204079|
|  Mobile|309.2663541666669|
+--------+-----------------+

+----------------+-----------------------+
|transaction_date|daily_transaction_count|
+----------------+-----------------------+
|      2024-01-15|                      7|
|      2024-01-16|                      9|
|      2024-01-17|                     10|
|      2024-01-18|                     10|
|      2024-01-19|                      9|
|      2024-01-20|                     10|
|      2024-01-21|                     10|
|      2024-01-22|                      8|
|      2024-01-23|                     10|
|      2024-01-24|                     10|
|      2024-01-25|        

In [ ]:
# TASK 6: Data Integration & Advanced Features

product_df=spark.read.csv("products.csv",header=True)

# Broadcast Join
joined_df = clean_df.join(
    broadcast(product_df),
    on="product_id",
    how="left"
).show()

+----------+--------------+--------+-------------+-------+--------+---------------------+--------+-------+--------------------+--------------+
|product_id|transaction_id| user_id|     category|  price|quantity|transaction_timestamp|platform|country|        product_name|         brand|
+----------+--------------+--------+-------------+-------+--------+---------------------+--------+-------+--------------------+--------------+
|   PROD072|        TXN172|USER1172|  electronics| 899.99|       1|  2024-02-01 09:15:30|     Web| France|Drone with Camera 4K|       SkyView|
|   PROD069|        TXN069|USER1069|  electronics|1099.99|       1|  2024-01-21 16:30:15|  Mobile| Canada|       Mouse Pad XXL|    GamingDesk|
|   PROD087|        TXN087|USER1087|      fashion|  115.0|       1|  2024-01-23 14:22:35|  Mobile| France|       Cardigan Knit|    LayerStyle|
|   PROD051|        TXN051|USER1051|       sports|  355.0|       1|  2024-01-20 08:30:45|  Mobile|    USA|      Skateboard Pro| ExtremeSports|

In [ ]:
# TASK 7: Accumulator for Tracking Invalid Records

invalid_counter = spark.sparkContext.accumulator(0)

def count_invalid(row):
    if (
        row.transaction_id is None or
        row.user_id is None or
        row.product_id is None
    ):
        invalid_counter.add(1)

df_raw.foreach(count_invalid)
print("Total Invalid Records:", invalid_counter.value)

Total Invalid Records: 7


In [ ]:
# TASK 7: Output Management (Final Dataset Write)

final_df = clean_df.withColumn(
    "transaction_date",
    to_date(col("transaction_timestamp"), "yyyy-MM-dd HH:mm:ss")
).filter(col("transaction_date").isNotNull())

final_df.write \
    .mode("overwrite") \
    .partitionBy("transaction_date") \
    .parquet("file:///C:/output/partitioned_transactions")